# BioNutriGPT — Multi-Modal Risk Architecture

**Colab + NVIDIA T4 implementation**

`Biomarkers → FT-Transformer → Z_bio`  
`Nutrition → SAINT-style Transformer → Z_nutri`  
`Lifestyle + Demographics → FT-Transformer → Z_life`  
`Z_bio + Z_nutri + Z_life → Cross-Modal Transformer → Z_fused`  
`Z_fused → Adaptive Mixture of Experts → Z_MoE`  
`Z_MoE → Diabetes / Obesity / CVD heads → probabilities + MC-Dropout uncertainty`

The notebook reads the CSV from Google Drive and uses only the columns required by each branch.

**Leakage control:** target columns are never inputs. BMI/weight/waist and disease-diagnosis questionnaire variables are excluded from the lifestyle branch.


In [6]:
# 1. Install dependencies
!pip -q install scikit-learn joblib


In [18]:
# 2. Imports

import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
)

import joblib

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.11.0+cu130
Device: cuda


In [7]:
# 3. Mount Drive and configure paths

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DATA_PATH = Path(
    "/content/drive/MyDrive/BioNutriGpt/BioNutriGPT_final_dataset_2005_2018.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)
print("Data exists:", DATA_PATH.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data path: /content/drive/MyDrive/BioNutriGpt/BioNutriGPT_final_dataset_2005_2018.csv
Output directory: /content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal
Data exists: True


## Required feature groups

### 1) Biomarkers → FT-Transformer
Laboratory/metabolic measurements.

### 2) Nutrition → SAINT-style Transformer
First-day 24-hour dietary recall variables.

### 3) Lifestyle + Demographics → FT-Transformer
Physical activity + sleep + smoking + alcohol + demographics.

Demographics are **not a fourth encoder**. They are part of `Z_life`.


In [8]:
# 4. Exact columns used by each model

ID_COL = "SEQN"
TARGET_COLS = ["TARGET_DIABETES", "TARGET_OBESITY", "TARGET_CVD"]

# ---------- Biomarkers ----------
BIOMARKER_COLS = [
    "LBXGH", "LBXGLU", "LBXIN", "LBDHDD",
    "LBXTR", "LBDLDL", "LBXAPB", "LBXTC"
]

# ---------- Nutrition ----------
NUTRITION_COLS = [
    "DR1TKCAL", "DR1TPROT", "DR1TCARB", "DR1TSUGR",
    "DR1TFIBE", "DR1TTFAT", "DR1TSFAT", "DR1TMFAT",
    "DR1TPFAT", "DR1TCHOL", "DR1TATOC", "DR1TRET",
    "DR1TVARA", "DR1TACAR", "DR1TBCAR", "DR1TLYCO",
    "DR1TLZ", "DR1TVB1", "DR1TVB2", "DR1TNIAC",
    "DR1TVB6", "DR1TFOLA", "DR1TFA", "DR1TFF",
    "DR1TFDFE", "DR1TVB12", "DR1TVC", "DR1TVK",
    "DR1TCALC", "DR1TPHOS", "DR1TMAGN", "DR1TIRON",
    "DR1TZINC", "DR1TCOPP", "DR1TSODI", "DR1TPOTA",
    "DR1TSELE", "DR1TCAFF", "DR1TALCO"
]

# ---------- Lifestyle ----------
LIFESTYLE_COLS = [
    "PAQ605", "PAQ610", "PAD615",
    "PAQ620", "PAQ625", "PAD630",
    "PAQ635", "PAQ640", "PAD645",
    "PAQ650", "PAQ655", "PAD660",
    "PAQ665", "PAQ670", "PAD675",
    "PAD680", "PAQ710", "PAQ715",

    "SLD010H", "SLD020M", "SLQ030", "SLQ040",
    "SLQ050", "SLQ060", "SLQ120",

    "SMQ020", "SMD030", "SMQ040", "SMQ050Q",
    "SMQ050U", "SMD055", "SMD057",

    "ALQ101", "ALQ110", "ALQ120Q", "ALQ120U",
    "ALQ130", "ALQ140Q", "ALQ140U", "ALQ150",
    "ALQ151"
]

# Demographics are integrated into the lifestyle/context encoder.
DEMO_NUM_COLS = ["RIDAGEYR", "INDHHINC", "INDFMPIR"]
DEMO_CAT_COLS = ["RIAGENDR", "RIDRETH3", "DMDEDUC2", "DMDMARTL"]

ALL_REQUIRED = (
    [ID_COL] + BIOMARKER_COLS + NUTRITION_COLS + LIFESTYLE_COLS +
    DEMO_NUM_COLS + DEMO_CAT_COLS + TARGET_COLS
)

print("Biomarkers:", len(BIOMARKER_COLS))
print("Nutrition:", len(NUTRITION_COLS))
print("Lifestyle:", len(LIFESTYLE_COLS))
print("Demographic numeric:", len(DEMO_NUM_COLS))
print("Demographic categorical:", len(DEMO_CAT_COLS))


Biomarkers: 8
Nutrition: 39
Lifestyle: 41
Demographic numeric: 3
Demographic categorical: 4


In [13]:
# 5. Load full dataset

import pandas as pd
import numpy as np

print("Loading dataset...")

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully")
print("Shape:", df.shape)

# Check required columns
available = set(df.columns)
missing = [c for c in ALL_REQUIRED if c not in available]

if missing:
    print("\nMissing required columns:")
    for c in missing:
        print(" -", c)
    raise ValueError("Required columns are missing from the dataset.")

print("All required columns are present.")

Loading dataset...
Dataset loaded successfully
Shape: (42143, 839)
All required columns are present.


In [14]:
# 6. Feature engineering

import pandas as pd
import numpy as np

# Make sure numeric columns are numeric
NUM_COLS = (
    BIOMARKER_COLS
    + NUTRITION_COLS
    + LIFESTYLE_COLS
    + DEMO_NUM_COLS
)

for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Make categorical demographic columns strings
for c in DEMO_CAT_COLS:
    if c in df.columns:
        df[c] = df[c].astype("string")


def add_lifestyle_features(frame):
    x = frame.copy()

    # Sleep duration
    x["SLEEP_HOURS_DERIVED"] = (
        pd.to_numeric(x["SLD010H"], errors="coerce")
        + pd.to_numeric(x["SLD020M"], errors="coerce") / 60.0
    )

    # Activity duration
    pairs = [
        ("PAQ610", "PAD615"),
        ("PAQ625", "PAD630"),
        ("PAQ640", "PAD645"),
        ("PAQ655", "PAD660"),
        ("PAQ670", "PAD675"),
    ]

    names = []

    for f, m in pairs:
        name = f"{f}_{m}_MIN"

        x[name] = (
            pd.to_numeric(x[f], errors="coerce")
            * pd.to_numeric(x[m], errors="coerce")
        )

        names.append(name)

    # Total activity minutes
    x["TOTAL_ACTIVITY_MIN_DERIVED"] = x[names].sum(
        axis=1,
        min_count=1
    )

    return x


df = add_lifestyle_features(df)

# Final lifestyle numeric feature list
LIFESTYLE_NUM_COLS = LIFESTYLE_COLS + [
    "SLEEP_HOURS_DERIVED",
    "PAQ610_PAD615_MIN",
    "PAQ625_PAD630_MIN",
    "PAQ640_PAD645_MIN",
    "PAQ655_PAD660_MIN",
    "PAQ670_PAD675_MIN",
    "TOTAL_ACTIVITY_MIN_DERIVED",
]

print("Lifestyle numeric:", len(LIFESTYLE_NUM_COLS))
print("Dataset shape:", df.shape)


Lifestyle numeric: 48
Dataset shape: (42143, 846)


In [16]:
# 7. Train / validation / test split
from sklearn.model_selection import train_test_split

# Remove rows where any target value is missing
before = len(df)

df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

print(f"Removed {before - len(df)} rows with missing target values")
print(f"Remaining rows: {len(df)}")

# Make sure targets are valid integers
stratify_code = (
    df[TARGET_COLS]
    .astype(int)
    .astype(str)
    .agg("".join, axis=1)
)

# First split: 70% train, 30% temporary
train_idx, temp_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.30,
    random_state=42,
    stratify=stratify_code
)

# Split temporary into 15% validation + 15% test
temp_stratify = stratify_code.iloc[temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_stratify
)

print("\nSplit sizes:")
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))


Removed 0 rows with missing target values
Remaining rows: 36729

Split sizes:
Train: 25710
Validation: 5509
Test: 5510


In [19]:
# 8. Fit preprocessing on TRAIN only
# This cell creates ALL arrays required by the multimodal Dataset.
# Missingness masks are captured BEFORE imputation.

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# -----------------------------
# Numeric feature groups
# -----------------------------
def prepare_numeric_group(columns, name):
    raw = df[columns].apply(pd.to_numeric, errors="coerce")
    train_raw = raw.iloc[train_idx]

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    train_imp = imputer.fit_transform(train_raw)
    scaler.fit(train_imp)

    full_missing = raw.isna().astype(np.float32).values
    full_imp = imputer.transform(raw)
    full_x = scaler.transform(full_imp).astype(np.float32)

    print(f"{name}: {full_x.shape}")
    return imputer, scaler, full_x, full_missing

bio_imp, bio_scaler, bio_x, bio_missing = prepare_numeric_group(
    BIOMARKER_COLS, "Biomarkers"
)

nutri_imp, nutri_scaler, nutri_x, nutri_missing = prepare_numeric_group(
    NUTRITION_COLS, "Nutrition"
)

life_imp, life_scaler, life_x, life_missing = prepare_numeric_group(
    LIFESTYLE_NUM_COLS, "Lifestyle"
)

# -----------------------------
# Demographic numeric features
# Kept as a separate preprocessed artifact for compatibility.
# The current architecture uses lifestyle numeric + categorical
# demographics in the lifestyle/context encoder.
# -----------------------------
demo_num_imp, demo_scaler, demo_num_x, demo_num_missing = prepare_numeric_group(
    DEMO_NUM_COLS, "Demographic numeric"
)

# -----------------------------
# Demographic categorical features
# -----------------------------
cat_data = df[DEMO_CAT_COLS].copy()

for c in DEMO_CAT_COLS:
    cat_data[c] = cat_data[c].astype(object)
    cat_data[c] = cat_data[c].where(pd.notna(cat_data[c]), np.nan)

cat_train = cat_data.iloc[train_idx]
cat_val = cat_data.iloc[val_idx]
cat_test = cat_data.iloc[test_idx]

cat_imp = SimpleImputer(strategy="most_frequent")

cat_train_imp = cat_imp.fit_transform(cat_train)
cat_val_imp = cat_imp.transform(cat_val)
cat_test_imp = cat_imp.transform(cat_test)

cat_enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

cat_train_encoded = cat_enc.fit_transform(cat_train_imp)
cat_val_encoded = cat_enc.transform(cat_val_imp)
cat_test_encoded = cat_enc.transform(cat_test_imp)

# Shift category indices by +1 so unknown=-1 becomes 0.
# This makes every categorical index valid for nn.Embedding.
cat_train_encoded = (cat_train_encoded + 1).astype(np.int64)
cat_val_encoded = (cat_val_encoded + 1).astype(np.int64)
cat_test_encoded = (cat_test_encoded + 1).astype(np.int64)

demo_cat_x = np.zeros(
    (len(df), len(DEMO_CAT_COLS)),
    dtype=np.int64
)
demo_cat_x[train_idx] = cat_train_encoded
demo_cat_x[val_idx] = cat_val_encoded
demo_cat_x[test_idx] = cat_test_encoded

# Targets
y = df[TARGET_COLS].astype(np.float32).values

print("\nPreprocessing complete")
print("bio_x:", bio_x.shape)
print("nutri_x:", nutri_x.shape)
print("life_x:", life_x.shape)
print("demo_cat_x:", demo_cat_x.shape)
print("y:", y.shape)


Biomarkers: (36729, 8)
Nutrition: (36729, 39)
Lifestyle: (36729, 48)
Demographic numeric: (36729, 3)

Preprocessing complete
bio_x: (36729, 8)
nutri_x: (36729, 39)
life_x: (36729, 48)
demo_cat_x: (36729, 4)
y: (36729, 3)


In [20]:
# 9. Dataset / DataLoaders

class MultiModalDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]

        return {
            "bio": torch.tensor(bio_x[idx], dtype=torch.float32),
            "bio_missing": torch.tensor(bio_missing[idx], dtype=torch.float32),

            "nutri": torch.tensor(nutri_x[idx], dtype=torch.float32),
            "nutri_missing": torch.tensor(nutri_missing[idx], dtype=torch.float32),

            "life": torch.tensor(life_x[idx], dtype=torch.float32),
            "life_missing": torch.tensor(life_missing[idx], dtype=torch.float32),

            "life_cat": torch.tensor(demo_cat_x[idx], dtype=torch.long),

            "y": torch.tensor(y[idx], dtype=torch.float32),
        }

BATCH_SIZE = 512

train_loader = DataLoader(
    MultiModalDataset(train_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    MultiModalDataset(val_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    MultiModalDataset(test_idx),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Multimodal DataLoaders created.")
print("Train:", len(train_loader.dataset))
print("Validation:", len(val_loader.dataset))
print("Test:", len(test_loader.dataset))


Multimodal DataLoaders created.
Train: 25710
Validation: 5509
Test: 5510


## Model blocks

The implementation below follows the requested architecture:

- **FT-Transformer** for biomarkers
- **SAINT-style tabular Transformer** for nutrition
- **FT-Transformer** for lifestyle + demographics
- **Cross-Modal Transformer** with multi-head attention
- **Adaptive MoE** with Metabolic/Nutrition/Lifestyle experts and a gating network
- **Three task heads**
- **MC-Dropout uncertainty**


In [21]:
# 10. Tokenizers and FT-Transformer

class NumericalTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))

    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class MissingnessEmbedding(nn.Module):
    def __init__(self, d_token):
        super().__init__()
        self.emb = nn.Embedding(2, d_token)

    def forward(self, missing):
        return self.emb(missing.long())

class CategoricalTokenizer(nn.Module):
    def __init__(self, cardinalities, d_token):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(int(card) + 1, d_token) for card in cardinalities
        ])

    def forward(self, x):
        return torch.cat(
            [emb(x[:, j]).unsqueeze(1) for j, emb in enumerate(self.embeddings)],
            dim=1
        )

class FTTransformerEncoder(nn.Module):
    def __init__(self, n_num, n_cat=0, cat_cardinalities=None,
                 d_token=64, n_heads=4, n_layers=3,
                 ff_dim=256, out_dim=128, dropout=0.15):
        super().__init__()
        self.num_tokenizer = NumericalTokenizer(n_num, d_token)
        self.num_missing = MissingnessEmbedding(d_token)
        self.n_cat = n_cat

        if n_cat:
            self.cat_tokenizer = CategoricalTokenizer(cat_cardinalities, d_token)

        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.normal_(self.cls, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.proj = nn.Sequential(
            nn.Linear(d_token, out_dim), nn.GELU(), nn.Dropout(dropout)
        )

    def forward(self, x_num, x_missing, x_cat=None):
        tokens = self.num_tokenizer(x_num) + self.num_missing(x_missing)

        if self.n_cat:
            tokens = torch.cat([tokens, self.cat_tokenizer(x_cat)], dim=1)

        cls = self.cls.expand(x_num.size(0), -1, -1)
        h = torch.cat([cls, tokens], dim=1)
        h = self.encoder(h)

        return self.proj(self.norm(h[:, 0]))


In [22]:
# 11. SAINT-style nutrition encoder

class SAINTStyleEncoder(nn.Module):
    def __init__(self, n_features, d_token=64, n_heads=4,
                 n_layers=3, ff_dim=256, out_dim=128, dropout=0.15):
        super().__init__()
        self.tokenizer = NumericalTokenizer(n_features, d_token)
        self.missing = MissingnessEmbedding(d_token)

        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.col_encoder = nn.TransformerEncoder(layer, num_layers=n_layers)

        self.row_mixer = nn.Sequential(
            nn.Linear(n_features * d_token, ff_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ff_dim, n_features * d_token)
        )

        self.norm = nn.LayerNorm(d_token)
        self.proj = nn.Sequential(
            nn.Linear(d_token, out_dim), nn.GELU(), nn.Dropout(dropout)
        )

    def forward(self, x, missing):
        h = self.tokenizer(x) + self.missing(missing)
        h = self.col_encoder(h)

        # Row-wise context mixing keeps the model T4-friendly.
        mixed = self.row_mixer(h.flatten(1)).view_as(h)
        h = h + mixed

        return self.proj(self.norm(h.mean(dim=1)))


In [23]:
# 12. Cross-Modal Transformer

class CrossModalTransformer(nn.Module):
    def __init__(self, d_in=128, d_model=256, n_heads=8, n_layers=2, dropout=0.15):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model)
        self.modality_embedding = nn.Parameter(torch.randn(1, 3, d_model) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bio, z_nutri, z_life):
        x = torch.stack([z_bio, z_nutri, z_life], dim=1)
        x = self.proj(x) + self.modality_embedding
        x = self.encoder(x)

        # Modality-aware pooling
        z_fused = self.norm(x.mean(dim=1))
        return z_fused, x


In [24]:
# 13. Adaptive Mixture of Experts

class Expert(nn.Module):
    def __init__(self, d_in=256, d_hidden=256, d_out=256, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_hidden, d_out), nn.GELU()
        )

    def forward(self, x):
        return self.net(x)

class AdaptiveMoE(nn.Module):
    def __init__(self, d_in=256, d_out=256, dropout=0.15):
        super().__init__()
        self.experts = nn.ModuleList([
            Expert(d_in, 256, d_out, dropout),
            Expert(d_in, 256, d_out, dropout),
            Expert(d_in, 256, d_out, dropout),
        ])

        self.gate = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, 128), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 3)
        )
        self.norm = nn.LayerNorm(d_out)

    def forward(self, z_fused):
        experts = torch.stack([e(z_fused) for e in self.experts], dim=1)
        weights = torch.softmax(self.gate(z_fused), dim=-1)
        z_moe = (experts * weights.unsqueeze(-1)).sum(dim=1)
        return self.norm(z_moe), weights


In [25]:
# 14. Complete BioNutriGPT model

class BioNutriGPT(nn.Module):
    def __init__(self, n_bio, n_nutri, n_life_num,
                 life_cat_cardinalities, d=128, fused_dim=256, dropout=0.15):
        super().__init__()

        self.bio_encoder = FTTransformerEncoder(
            n_num=n_bio, d_token=64, n_heads=4, n_layers=3,
            ff_dim=256, out_dim=d, dropout=dropout
        )

        self.nutri_encoder = SAINTStyleEncoder(
            n_features=n_nutri, d_token=64, n_heads=4, n_layers=3,
            ff_dim=256, out_dim=d, dropout=dropout
        )

        self.life_encoder = FTTransformerEncoder(
            n_num=n_life_num,
            n_cat=len(life_cat_cardinalities),
            cat_cardinalities=life_cat_cardinalities,
            d_token=64, n_heads=4, n_layers=3,
            ff_dim=256, out_dim=d, dropout=dropout
        )

        self.cross_modal = CrossModalTransformer(
            d_in=d, d_model=fused_dim, n_heads=8, n_layers=2, dropout=dropout
        )

        self.moe = AdaptiveMoE(fused_dim, fused_dim, dropout)

        self.diabetes_head = nn.Linear(fused_dim, 1)
        self.obesity_head = nn.Linear(fused_dim, 1)
        self.cvd_head = nn.Linear(fused_dim, 1)

    def forward(self, b):
        z_bio = self.bio_encoder(b["bio"], b["bio_missing"])
        z_nutri = self.nutri_encoder(b["nutri"], b["nutri_missing"])
        z_life = self.life_encoder(b["life"], b["life_missing"], b["life_cat"])

        z_fused, modality_tokens = self.cross_modal(z_bio, z_nutri, z_life)
        z_moe, gate_weights = self.moe(z_fused)

        logits = torch.cat([
            self.diabetes_head(z_moe),
            self.obesity_head(z_moe),
            self.cvd_head(z_moe)
        ], dim=1)

        return {
            "z_bio": z_bio, "z_nutri": z_nutri, "z_life": z_life,
            "z_fused": z_fused, "z_moe": z_moe,
            "modality_tokens": modality_tokens,
            "gate_weights": gate_weights,
            "logits": logits
        }


In [26]:
# 15. Instantiate model

life_cat_cardinalities = [int(len(c)) + 1 for c in cat_enc.categories_]

model = BioNutriGPT(
    n_bio=len(BIOMARKER_COLS),
    n_nutri=len(NUTRITION_COLS),
    n_life_num=life_x.shape[1],
    life_cat_cardinalities=life_cat_cardinalities,
    d=128,
    fused_dim=256,
    dropout=0.15
).to(device)

print(
    "Parameters:",
    round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "M"
)


Parameters: 3.81 M


In [27]:
# 16. Loss, optimizer, scheduler and checkpoint

y_train = y[train_idx]

pos = y_train.sum(axis=0)
neg = len(y_train) - pos

pos_weight = np.divide(
    neg,
    pos,
    out=np.ones_like(pos, dtype=np.float32),
    where=pos > 0
)

pos_weight = torch.tensor(
    pos_weight,
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

MODEL_DIR = OUTPUT_DIR
best_path = MODEL_DIR / "best_bionutrigt_multimodal.pt"

best_val_auc = -np.inf
history = []

print("pos_weight:", pos_weight.detach().cpu().numpy())
print("Checkpoint:", best_path)


pos_weight: [6.640416  1.6573644 9.404695 ]
Checkpoint: /content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal/best_bionutrigt_multimodal.pt


In [28]:
# 16A. Sanity-check one batch

batch = next(iter(train_loader))

print(batch.keys())
for k, v in batch.items():
    print(f"{k:15s}", tuple(v.shape), v.dtype)

with torch.no_grad():
    batch_device = {k: v.to(device) for k, v in batch.items()}
    output = model(batch_device)

print("Forward pass successful")
print("Logits shape:", tuple(output["logits"].shape))


dict_keys(['bio', 'bio_missing', 'nutri', 'nutri_missing', 'life', 'life_missing', 'life_cat', 'y'])
bio             (512, 8) torch.float32
bio_missing     (512, 8) torch.float32
nutri           (512, 39) torch.float32
nutri_missing   (512, 39) torch.float32
life            (512, 48) torch.float32
life_missing    (512, 48) torch.float32
life_cat        (512, 4) torch.int64
y               (512, 3) torch.float32
Forward pass successful
Logits shape: (512, 3)


In [29]:
# 17. Train / Evaluate

from sklearn.metrics import roc_auc_score

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    n = 0
    all_p = []
    all_y = []

    for batch in loader:
        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            output = model(batch)
            logits = output["logits"]

            loss = criterion(logits, batch["y"])

            if train:
                loss.backward()
                optimizer.step()

        bs = batch["y"].size(0)
        total_loss += loss.item() * bs
        n += bs

        probs = torch.sigmoid(logits)

        all_p.append(probs.detach().cpu().numpy())
        all_y.append(batch["y"].detach().cpu().numpy())

    all_p = np.concatenate(all_p, axis=0)
    all_y = np.concatenate(all_y, axis=0)

    avg_loss = total_loss / n

    aucs = []
    for j in range(all_y.shape[1]):
        try:
            auc = roc_auc_score(all_y[:, j], all_p[:, j])
        except ValueError:
            auc = np.nan
        aucs.append(auc)

    mean_auc = np.nanmean(aucs)

    return avg_loss, mean_auc, aucs, all_p, all_y

print("run_epoch() ready")


run_epoch() ready


In [31]:
# 17A. Train for 75 epochs and save the BEST validation model

EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_auc, _, _, _ = run_epoch(train_loader, train=True)

    va_loss, va_auc, va_aucs, _, _ = run_epoch(
        val_loader,
        train=False
    )

    scheduler.step(va_auc)

    history.append([
        epoch,
        tr_loss,
        tr_auc,
        va_loss,
        va_auc,
        va_aucs[0],
        va_aucs[1],
        va_aucs[2]
    ])

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {tr_loss:.4f} | "
        f"Train AUC: {tr_auc:.4f} | "
        f"Val Loss: {va_loss:.4f} | "
        f"Val AUC: {va_auc:.4f}"
    )

    print(
        f"  Diabetes: {va_aucs[0]:.4f} | "
        f"Obesity: {va_aucs[1]:.4f} | "
        f"CVD: {va_aucs[2]:.4f}"
    )

    if np.isfinite(va_auc) and va_auc > best_val_auc:
        best_val_auc = va_auc

        torch.save(
            {
                "model_state": model.state_dict(),
                "best_val_auc": float(best_val_auc),
                "epoch": epoch
            },
            best_path
        )

        print("  ✓ Best model saved:", best_path)

print("\nTraining complete.")
print("Best validation AUC:", best_val_auc)
print("Best checkpoint:", best_path)


Epoch 01/50 | Train Loss: 0.7103 | Train AUC: 0.8476 | Val Loss: 0.7625 | Val AUC: 0.8339
  Diabetes: 0.9393 | Obesity: 0.7666 | CVD: 0.7959
Epoch 02/50 | Train Loss: 0.7114 | Train AUC: 0.8473 | Val Loss: 0.7619 | Val AUC: 0.8339
  Diabetes: 0.9395 | Obesity: 0.7665 | CVD: 0.7959
Epoch 03/50 | Train Loss: 0.7095 | Train AUC: 0.8482 | Val Loss: 0.7629 | Val AUC: 0.8340
  Diabetes: 0.9396 | Obesity: 0.7665 | CVD: 0.7961
Epoch 04/50 | Train Loss: 0.7102 | Train AUC: 0.8478 | Val Loss: 0.7627 | Val AUC: 0.8341
  Diabetes: 0.9396 | Obesity: 0.7665 | CVD: 0.7961
Epoch 05/50 | Train Loss: 0.7098 | Train AUC: 0.8481 | Val Loss: 0.7624 | Val AUC: 0.8340
  Diabetes: 0.9396 | Obesity: 0.7664 | CVD: 0.7961
Epoch 06/50 | Train Loss: 0.7098 | Train AUC: 0.8477 | Val Loss: 0.7629 | Val AUC: 0.8341
  Diabetes: 0.9395 | Obesity: 0.7665 | CVD: 0.7961
Epoch 07/50 | Train Loss: 0.7099 | Train AUC: 0.8475 | Val Loss: 0.7628 | Val AUC: 0.8340
  Diabetes: 0.9396 | Obesity: 0.7665 | CVD: 0.7961
Epoch 08/50 |

In [32]:
# 18. Test metrics using the BEST checkpoint

if not best_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {best_path}")

ckpt = torch.load(
    best_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(ckpt["model_state"])
model.eval()

test_loss, test_auc, test_aucs, test_p, test_y = run_epoch(
    test_loader,
    train=False
)

print("Best epoch:", ckpt["epoch"])
print("Best validation AUC:", ckpt["best_val_auc"])
print("Test loss:", test_loss)
print("Mean ROC-AUC:", test_auc)

for name, auc in zip(TARGET_COLS, test_aucs):
    print(name, "ROC-AUC:", round(float(auc), 4))


Best epoch: 52
Best validation AUC: 0.8342818663282326
Test loss: 0.7466574267769899
Mean ROC-AUC: 0.8366847602384494
TARGET_DIABETES ROC-AUC: 0.938
TARGET_OBESITY ROC-AUC: 0.7533
TARGET_CVD ROC-AUC: 0.8188


In [33]:
# 19. MC-Dropout uncertainty
def enable_dropout(m):
    if isinstance(m, nn.Dropout):
        m.train()

@torch.no_grad()
def mc_predict(loader, passes=30):
    model.eval()
    model.apply(enable_dropout)

    samples = []

    for _ in range(passes):
        preds = []
        for batch in loader:
            batch = {k:v.to(device, non_blocking=True) for k,v in batch.items()}
            out = model(batch)
            preds.append(torch.sigmoid(out["logits"]).cpu().numpy())
        samples.append(np.concatenate(preds))

    samples = np.stack(samples)
    mean = samples.mean(axis=0)
    std = samples.std(axis=0)
    lo = np.percentile(samples, 2.5, axis=0)
    hi = np.percentile(samples, 97.5, axis=0)

    return mean, std, lo, hi

mc_mean, mc_std, mc_lo, mc_hi = mc_predict(test_loader, passes=30)

for j, name in enumerate(["DIABETES", "OBESITY", "CVD"]):
    print(
        name,
        "mean probability =", round(float(mc_mean[:,j].mean()), 4),
        "mean uncertainty =", round(float(mc_std[:,j].mean()), 4)
    )

model.eval()


DIABETES mean probability = 0.2557 mean uncertainty = 0.0161
OBESITY mean probability = 0.4764 mean uncertainty = 0.0272
CVD mean probability = 0.342 mean uncertainty = 0.0233


BioNutriGPT(
  (bio_encoder): FTTransformerEncoder(
    (num_tokenizer): NumericalTokenizer()
    (num_missing): MissingnessEmbedding(
      (emb): Embedding(2, 64)
    )
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.15, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.15, inplace=False)
          (dropout2): Dropout(p=0.15, inplace=False)
        )
      )
    )
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (proj): Sequential(
      

In [34]:
# 20. Save risk predictions + uncertainty
test_ids = df.iloc[test_idx][ID_COL].values

results = pd.DataFrame({
    "SEQN": test_ids,

    "DIABETES_PROB": mc_mean[:,0],
    "DIABETES_UNCERTAINTY": mc_std[:,0],
    "DIABETES_LOWER_95": mc_lo[:,0],
    "DIABETES_UPPER_95": mc_hi[:,0],

    "OBESITY_PROB": mc_mean[:,1],
    "OBESITY_UNCERTAINTY": mc_std[:,1],
    "OBESITY_LOWER_95": mc_lo[:,1],
    "OBESITY_UPPER_95": mc_hi[:,1],

    "CVD_PROB": mc_mean[:,2],
    "CVD_UNCERTAINTY": mc_std[:,2],
    "CVD_LOWER_95": mc_lo[:,2],
    "CVD_UPPER_95": mc_hi[:,2],
})

pred_path = OUTPUT_DIR / "risk_predictions_with_uncertainty.csv"
results.to_csv(pred_path, index=False)
print(pred_path)
results.head()


/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal/risk_predictions_with_uncertainty.csv


,SEQN,DIABETES_PROB,DIABETES_UNCERTAINTY,DIABETES_LOWER_95,DIABETES_UPPER_95,OBESITY_PROB,OBESITY_UNCERTAINTY,OBESITY_LOWER_95,OBESITY_UPPER_95,CVD_PROB,CVD_UNCERTAINTY,CVD_LOWER_95,CVD_UPPER_95
0,100144.0,0.090612,0.011273,0.071413,0.109696,0.562972,0.038798,0.492338,0.634179,0.233061,0.019225,0.193880,0.266994
1,76169.0,0.032096,0.006353,0.022037,0.043957,0.581985,0.024107,0.539414,0.619202,0.193979,0.022896,0.158471,0.237848
2,77340.0,0.006718,0.001124,0.005490,0.009224,0.363479,0.032229,0.301081,0.413799,0.052310,0.006766,0.040037,0.066956
3,102948.0,0.018179,0.002442,0.014777,0.023388,0.303293,0.033391,0.257755,0.368001,0.178092,0.012772,0.151500,0.201959
4,46525.0,0.019485,0.004472,0.013314,0.028794,0.608511,0.034778,0.544307,0.670062,0.034179,0.017147,0.017676,0.083747


In [35]:
# 21. Extract Z_bio, Z_nutri, Z_life, Z_fused, Z_MoE and gating weights
@torch.no_grad()
def extract_embeddings(loader):
    model.eval()
    rows = []

    for batch in loader:
        b = {k:v.to(device, non_blocking=True) for k,v in batch.items()}
        out = model(b)

        for i in range(b["y"].shape[0]):
            rows.append({
                "z_bio": out["z_bio"][i].cpu().numpy(),
                "z_nutri": out["z_nutri"][i].cpu().numpy(),
                "z_life": out["z_life"][i].cpu().numpy(),
                "z_fused": out["z_fused"][i].cpu().numpy(),
                "z_moe": out["z_moe"][i].cpu().numpy(),
                "gate_metabolic": out["gate_weights"][i,0].item(),
                "gate_nutrition": out["gate_weights"][i,1].item(),
                "gate_lifestyle": out["gate_weights"][i,2].item()
            })
    return rows

rows = extract_embeddings(test_loader)

emb = pd.DataFrame({
    "SEQN": test_ids,
    "gate_metabolic": [r["gate_metabolic"] for r in rows],
    "gate_nutrition": [r["gate_nutrition"] for r in rows],
    "gate_lifestyle": [r["gate_lifestyle"] for r in rows]
})

for key in ["z_bio", "z_nutri", "z_life", "z_fused", "z_moe"]:
    a = np.stack([r[key] for r in rows])
    for j in range(a.shape[1]):
        emb[f"{key}_{j}"] = a[:,j]

emb_path = OUTPUT_DIR / "test_embeddings_and_moe_gates.csv"
emb.to_csv(emb_path, index=False)
print(emb_path)


/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal/test_embeddings_and_moe_gates.csv


In [36]:
# 22. Save preprocessing and architecture metadata

import joblib

joblib.dump(
    {
        "bio_imputer": bio_imp,
        "bio_scaler": bio_scaler,
        "nutri_imputer": nutri_imp,
        "nutri_scaler": nutri_scaler,
        "life_imputer": life_imp,
        "life_scaler": life_scaler,
        "demo_imputer": demo_num_imp,
        "demo_scaler": demo_scaler,
        "cat_imputer": cat_imp,
        "cat_encoder": cat_enc,
        "BIOMARKER_COLS": BIOMARKER_COLS,
        "NUTRITION_COLS": NUTRITION_COLS,
        "LIFESTYLE_NUM_COLS": LIFESTYLE_NUM_COLS,
        "DEMO_NUM_COLS": DEMO_NUM_COLS,
        "DEMO_CAT_COLS": DEMO_CAT_COLS,
        "TARGET_COLS": TARGET_COLS
    },
    OUTPUT_DIR / "preprocessing.joblib"
)

history_df = pd.DataFrame(
    history,
    columns=[
        "epoch",
        "train_loss",
        "train_auc",
        "val_loss",
        "val_auc",
        "val_diabetes_auc",
        "val_obesity_auc",
        "val_cvd_auc"
    ]
)

history_df.to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

with open(OUTPUT_DIR / "architecture.json", "w") as f:
    json.dump(
        {
            "biomarker_encoder": "FT-Transformer",
            "nutrition_encoder": "SAINT-style tabular Transformer",
            "lifestyle_demographic_encoder": "FT-Transformer",
            "cross_modal_transformer": "2-layer 8-head Transformer",
            "fusion_dimension": 256,
            "moe_experts": [
                "Metabolic",
                "Nutrition",
                "Lifestyle"
            ],
            "risk_heads": [
                "Diabetes",
                "Obesity",
                "CVD"
            ],
            "uncertainty": "MC-Dropout",
            "epochs": 75,
            "best_checkpoint": "best_bionutrigt_multimodal.pt"
        },
        f,
        indent=2
    )

print("All artifacts saved in:", OUTPUT_DIR)


All artifacts saved in: /content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal


In [37]:
# 23. Check saved artifacts

for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name, "-", p.stat().st_size, "bytes")


architecture.json - 479 bytes
best_bionutrigt_multimodal.pt - 15335443 bytes
preprocessing.joblib - 9971 bytes
risk_predictions_with_uncertainty.csv - 781649 bytes
test_embeddings_and_moe_gates.csv - 55778298 bytes
training_history.csv - 15464 bytes


In [38]:
# ============================================================
# BIO NUTRI GPT — BLOOD REPORT ONLY TEST
# ============================================================
# Enter ONLY these 8 blood-report values below.
# No other user input is required.
# ============================================================

import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path


# ============================================================
# 1. BLOOD REPORT INPUTS
# ============================================================

blood_report = {

    # HbA1c (%)
    "LBXGH": 8.3,

    # Blood glucose (mg/dL)
    "LBXGLU": 180.0,

    # Insulin
    "LBXIN": np.nan,

    # HDL cholesterol (mg/dL)
    "LBDHDD": 28.0,

    # Triglycerides (mg/dL)
    "LBXTR": 259.0,

    # LDL cholesterol (mg/dL)
    "LBDLDL": 98.0,

    # Apolipoprotein B
    "LBXAPB": np.nan,

    # Total cholesterol (mg/dL)
    "LBXTC": 214.0
}


# ============================================================
# 2. LOAD PREPROCESSING
# ============================================================

PREP_PATH = OUTPUT_DIR / "preprocessing.joblib"

prep = joblib.load(PREP_PATH)


# ============================================================
# 3. GET TRAINING FEATURE LISTS
# ============================================================

BIOMARKER_COLS = prep["BIOMARKER_COLS"]
NUTRITION_COLS = prep["NUTRITION_COLS"]
LIFESTYLE_NUM_COLS = prep["LIFESTYLE_NUM_COLS"]
DEMO_NUM_COLS = prep["DEMO_NUM_COLS"]
DEMO_CAT_COLS = prep["DEMO_CAT_COLS"]


# ============================================================
# 4. GET PREPROCESSORS
# ============================================================

bio_imputer = prep["bio_imputer"]
bio_scaler = prep["bio_scaler"]

nutri_imputer = prep["nutri_imputer"]
nutri_scaler = prep["nutri_scaler"]

life_imputer = prep["life_imputer"]
life_scaler = prep["life_scaler"]

cat_imputer = prep["cat_imputer"]
cat_encoder = prep["cat_encoder"]


# ============================================================
# 5. CREATE EMPTY PATIENT
# ============================================================

patient = {}


# ------------------------------------------------------------
# Blood biomarkers
# ------------------------------------------------------------

for col in BIOMARKER_COLS:
    patient[col] = np.nan

# Insert only blood-report values
for col, value in blood_report.items():
    patient[col] = value


# ------------------------------------------------------------
# Everything else is missing
# ------------------------------------------------------------

for col in NUTRITION_COLS:
    patient[col] = np.nan

for col in LIFESTYLE_NUM_COLS:
    patient[col] = np.nan

for col in DEMO_NUM_COLS:
    patient[col] = np.nan

for col in DEMO_CAT_COLS:
    patient[col] = np.nan


# ============================================================
# 6. DATAFRAME
# ============================================================

patient_df = pd.DataFrame([patient])


# ============================================================
# 7. BIOMARKER INPUT
# ============================================================

bio_input = patient_df[
    BIOMARKER_COLS
].copy()


# ============================================================
# 8. NUTRITION INPUT
# ============================================================

nutri_input = patient_df[
    NUTRITION_COLS
].copy()


# ============================================================
# 9. LIFESTYLE INPUT
# ============================================================

life_input = patient_df[
    LIFESTYLE_NUM_COLS
].copy()


# ============================================================
# 10. DEMOGRAPHIC INPUT
# ============================================================

cat_input = patient_df[
    DEMO_CAT_COLS
].copy()


# ============================================================
# 11. MISSINGNESS MASKS
# ============================================================

bio_missing = (
    bio_input
    .isna()
    .astype(np.float32)
    .values
)

nutri_missing = (
    nutri_input
    .isna()
    .astype(np.float32)
    .values
)

life_missing = (
    life_input
    .isna()
    .astype(np.float32)
    .values
)


# ============================================================
# 12. PREPROCESS BLOOD
# ============================================================

bio_processed = bio_scaler.transform(
    bio_imputer.transform(
        bio_input
    )
)


# ============================================================
# 13. PREPROCESS NUTRITION
# ============================================================

nutri_processed = nutri_scaler.transform(
    nutri_imputer.transform(
        nutri_input
    )
)


# ============================================================
# 14. PREPROCESS LIFESTYLE
# ============================================================

life_processed = life_scaler.transform(
    life_imputer.transform(
        life_input
    )
)


# ============================================================
# 15. PREPROCESS CATEGORICAL
# ============================================================

cat_processed = cat_encoder.transform(
    cat_imputer.transform(
        cat_input
    )
)

# Match the +1 category indexing used during training.
cat_processed = (cat_processed + 1).astype(np.int64)


# ============================================================
# 16. TORCH
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

bio_tensor = torch.tensor(
    bio_processed,
    dtype=torch.float32,
    device=device
)

nutri_tensor = torch.tensor(
    nutri_processed,
    dtype=torch.float32,
    device=device
)

life_tensor = torch.tensor(
    life_processed,
    dtype=torch.float32,
    device=device
)

life_cat_tensor = torch.tensor(
    cat_processed,
    dtype=torch.long,
    device=device
)

bio_missing_tensor = torch.tensor(
    bio_missing,
    dtype=torch.float32,
    device=device
)

nutri_missing_tensor = torch.tensor(
    nutri_missing,
    dtype=torch.float32,
    device=device
)

life_missing_tensor = torch.tensor(
    life_missing,
    dtype=torch.float32,
    device=device
)


# ============================================================
# 17. MODEL INPUT
# ============================================================

model_input = {

    "bio":
        bio_tensor,

    "bio_missing":
        bio_missing_tensor,

    "nutri":
        nutri_tensor,

    "nutri_missing":
        nutri_missing_tensor,

    "life":
        life_tensor,

    "life_missing":
        life_missing_tensor,

    "life_cat":
        life_cat_tensor
}


# ============================================================
# 18. RUN MODEL
# ============================================================

# Always load the best 75-epoch checkpoint before this test.
BEST_PATH = OUTPUT_DIR / "best_bionutrigt_multimodal.pt"

if not BEST_PATH.exists():
    raise FileNotFoundError(
        f"Best model not found: {BEST_PATH}. Run the training section first."
    )

checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False
)

model = model.to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

with torch.no_grad():
    output = model(model_input)


# ============================================================
# 19. GET LOGITS
# ============================================================

logits = output["logits"]


# ============================================================
# 20. EXTRACT TASK OUTPUTS
# ============================================================

if isinstance(logits, dict):

    diabetes_logits = logits.get(
        "TARGET_DIABETES",
        logits.get("diabetes")
    )

    obesity_logits = logits.get(
        "TARGET_OBESITY",
        logits.get("obesity")
    )

    cvd_logits = logits.get(
        "TARGET_CVD",
        logits.get("cvd")
    )

elif isinstance(logits, (list, tuple)):

    diabetes_logits = logits[0]
    obesity_logits = logits[1]
    cvd_logits = logits[2]

else:

    diabetes_logits = logits[:, 0]
    obesity_logits = logits[:, 1]
    cvd_logits = logits[:, 2]


# ============================================================
# 21. PROBABILITY
# ============================================================

def probability(x):

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.detach()

    if x.ndim > 1 and x.shape[-1] == 1:
        x = x.squeeze(-1)

    return (
        torch.sigmoid(x)
        .cpu()
        .item()
        * 100
    )


diabetes_prob = probability(
    diabetes_logits
)

obesity_prob = probability(
    obesity_logits
)

cvd_prob = probability(
    cvd_logits
)


# ============================================================
# 22. DISPLAY BLOOD INPUTS
# ============================================================

print()
print("=" * 70)
print("BLOOD REPORT INPUTS")
print("=" * 70)

print(f"HbA1c                 : {blood_report['LBXGH']} %")
print(f"Blood glucose         : {blood_report['LBXGLU']} mg/dL")
print(f"Insulin               : {blood_report['LBXIN']}")
print(f"HDL cholesterol       : {blood_report['LBDHDD']} mg/dL")
print(f"Triglycerides         : {blood_report['LBXTR']} mg/dL")
print(f"LDL cholesterol       : {blood_report['LBDLDL']} mg/dL")
print(f"ApoB                  : {blood_report['LBXAPB']}")
print(f"Total cholesterol     : {blood_report['LBXTC']} mg/dL")


# ============================================================
# 23. MODEL OUTPUT
# ============================================================

print()
print("=" * 70)
print("BIO NUTRI GPT PREDICTION")
print("=" * 70)

print(
    f"Diabetes probability : "
    f"{diabetes_prob:.2f}%"
)

print(
    f"Obesity probability  : "
    f"{obesity_prob:.2f}%"
)

print(
    f"CVD probability      : "
    f"{cvd_prob:.2f}%"
)


# ============================================================
# 24. INPUT COVERAGE
# ============================================================

available_biomarkers = sum(
    pd.notna(v)
    for v in blood_report.values()
)

print()
print("=" * 70)
print("BLOOD INPUT COVERAGE")
print("=" * 70)

print(
    f"Available biomarkers : "
    f"{available_biomarkers}/8"
)

print(
    f"Missing biomarkers   : "
    f"{8 - available_biomarkers}/8"
)

print()
print(
    "Nutrition, lifestyle and demographic inputs were left "
    "missing and handled by the trained preprocessing pipeline."
)

print("=" * 70)


BLOOD REPORT INPUTS
HbA1c                 : 8.3 %
Blood glucose         : 180.0 mg/dL
Insulin               : nan
HDL cholesterol       : 28.0 mg/dL
Triglycerides         : 259.0 mg/dL
LDL cholesterol       : 98.0 mg/dL
ApoB                  : nan
Total cholesterol     : 214.0 mg/dL

BIO NUTRI GPT PREDICTION
Diabetes probability : 98.24%
Obesity probability  : 75.45%
CVD probability      : 79.78%

BLOOD INPUT COVERAGE
Available biomarkers : 6/8
Missing biomarkers   : 2/8

Nutrition, lifestyle and demographic inputs were left missing and handled by the trained preprocessing pipeline.


## Final output

The notebook implements exactly this stage:

```text
BIOMARKERS ── FT-Transformer ── Z_bio ───────────┐
                                                 │
NUTRITION ──── SAINT ─────────── Z_nutri ────────┼── Cross-Modal Transformer
                                                 │
LIFESTYLE + DEMOGRAPHICS ─ FT-Transformer ─ Z_life┘
                                                        ↓
                                                     Z_fused
                                                        ↓
                                                  Adaptive MoE
                                       ┌────────────┼────────────┐
                                  Metabolic      Nutrition      Lifestyle
                                    Expert        Expert          Expert
                                       └────────────┼────────────┘
                                                    ↓
                                                  Z_MoE
                                                    ↓
                                ┌───────────────────┼───────────────────┐
                                ↓                   ↓                   ↓
                            Diabetes             Obesity               CVD
                              Head                 Head                Head
                                ↓                   ↓                   ↓
                                      Probability + MC-Dropout
                                                ↓
                                   Risk + Uncertainty

Saved artifacts:
- best_bionutrigt_multimodal.pt
- risk_predictions_with_uncertainty.csv
- test_embeddings_and_moe_gates.csv
- preprocessing.joblib
- training_history.csv
- architecture.json
